# Classes & Objects — Inventing the Model API

So far your programs kept *data* (lists, dicts) and *behavior* (functions) in
separate boxes, and you passed one to the other by hand. A **class** glues them
together: an object carries its own data *and* the functions that work on it.

Why should a machine-learning book care? Because a trained model **is** an object —
some learned numbers plus a `predict` function that uses them. By the end of this
chapter you will have invented the `fit`/`predict` interface that scikit-learn,
Keras and friends all use, hand-built a decision tree out of objects, and saved a
model to disk.

---
## Part 1 — From Dict to Class

You could describe a customer with a dict:

```python
customer = {"name": "Sandeep", "place": "India"}
```

It works, but the data has no behavior: any code that wants to *do* something with
a customer must live somewhere else.

### Exercise 1.1 — `Customer`

Write a class `Customer`:

- `Customer("Sandeep", "India")` stores the two values on the object
  (**hint:** define `__init__(self, name, place)` and assign to `self.name`, `self.place`).
- A method `describe()` returns `"<name> from <place>"`.

**Write your solution in `exercises/ex_1_1_customer.py`**

In [ ]:
class Customer:
    pass  # replace this


c = Customer("Sandeep", "India")
assert c.name == "Sandeep"
assert c.place == "India"
assert c.describe() == "Sandeep from India"
c2 = Customer("Nitin", "Bharat")
assert c2.describe() == "Nitin from Bharat"
print("Customer: OK")

### What just happened?

- `Customer("Sandeep", "India")` **constructs** an object: Python creates an empty
  object and calls your `__init__` with it as `self`.
- `c.describe()` is exactly `Customer.describe(c)` — the object slides in as `self`.
  Try it: `Customer.describe(c)` returns the same string.
- `c` and `c2` each carry their **own** `name` and `place`. One class, many objects.

---
## Part 2 — A Model Is an Object: `TwoPointLine`

Remember the thermometer from *Expressions & Functions*? Calibration data:
current 1 → 30°, current 2 → 35°. What temperature is current 3?

You solved it with two loose functions, `fit` and `predict`, and had to carry the
`(m, c)` tuple between them yourself. Let the object carry it.

### Exercise 2.1 — `TwoPointLine`

Write a class `TwoPointLine` with:

- `fit(x, y)` — `x` and `y` are lists of **two** values each: two points
  `(x[0], y[0])` and `(x[1], y[1])`. Compute the slope
  `m = (y2 - y1) / (x2 - x1)` and intercept `c = y2 - m * x2`, and store them
  on `self`.
- `predict(xs)` — take a *list* of x values, return the list of `m * x + c`.

```python
model = TwoPointLine()
model.fit([2, 5], [3, 9])
model.predict([3])       # [5.0]
```

**Write your solution in `exercises/ex_2_1_twopointline.py`**

In [ ]:
class TwoPointLine:
    pass  # replace this


model = TwoPointLine()
model.fit([2, 5], [3, 9])            # the points (2, 3) and (5, 9)
assert model.predict([3, 0, 5]) == [5.0, -1.0, 9.0]

thermo = TwoPointLine()
thermo.fit([1, 2], [30, 35])         # the thermometer calibration
assert thermo.predict([3]) == [40.0]
assert thermo.predict([2.1]) == [35.5]

# two models live side by side without interfering:
assert model.predict([2]) == [3.0]
print("TwoPointLine: OK")

### You just invented the scikit-learn API

```python
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X, y)
model.predict(X_new)
```

Same shape: construct → `fit` stores learned parameters on the object →
`predict` uses them. Every model in the rest of this book fits this mold; the only
thing that changes is *how* `fit` finds the parameters.

### Exercise 2.2 — Guard the Gate

`fit` silently misbehaves if you hand it three points. Make it fail loudly instead:
copy your class below and make `fit` **raise** a `ValueError` unless both lists
have exactly two values (**hint:** `raise ValueError("need exactly two points")`).

**Write your solution in `exercises/ex_2_2_twopointline.py`**

In [ ]:
class TwoPointLine:
    pass  # replace this (copy your class from above and add validation)


model = TwoPointLine()
try:
    model.fit([1, 2, 3], [1, 2, 3])
    assert False, "fit should have raised ValueError"
except ValueError:
    pass

model.fit([2, 5], [3, 9])
assert model.predict([2]) == [3.0]
print("validation: OK")

---
## Part 3 — Inheritance: Classes That Extend Classes

All animals eat; only dogs bark. Instead of repeating `eat` in every animal class,
write it once and **inherit** it.

### Exercise 3.1 — `Animal`, `Dog`, `Cat`

- `Animal` has a method `eat()` returning `"eating"`.
- `Dog(Animal)` adds `bark()` returning `"woof!"` — and must **not** redefine `eat`.
- `Cat(Animal)` adds `meow()` returning `"meow"`.

The class in parentheses — `class Dog(Animal):` — is the parent. Everything the
parent can do, the child can do too.

**Write your solution in `exercises/ex_3_1_animal.py`**

In [ ]:
class Animal:
    pass  # replace this

class Dog(Animal):
    pass  # replace this

class Cat(Animal):
    pass  # replace this


d = Dog()
assert d.eat() == "eating"          # inherited: Dog never defines eat
assert d.bark() == "woof!"
assert Cat().eat() == "eating"
assert Cat().meow() == "meow"
assert isinstance(d, Animal)        # a Dog IS an Animal
print("inheritance: OK")

---
## Part 4 — Objects That Contain Objects: Trees

An object's attributes can be *other objects* — that is how you build trees.
Here is a company's expense tree: each node is a department with its own expense,
and (up to) two sub-departments:

```
        10
      /    \
     4      5
    / \    /  \
   1  11  15  16
```

### Exercise 4.1 — `Node` with `total` and `count`

Write a class `Node`:

- `Node(val, left=None, right=None)` stores the value and the two children.
- `total()` returns this node's `val` plus the totals of its children
  (skip a child if it is `None`).
- `count()` returns how many nodes are in this subtree.

Notice the recursion: a method calling *the same method on its children*.

**Write your solution in `exercises/ex_4_1_node.py`**

In [ ]:
class Node:
    pass  # replace this


tree = Node(10,
            Node(4, Node(1), Node(11)),
            Node(5, Node(15), Node(16)))
assert tree.total() == 62
assert tree.count() == 7
assert Node(42).total() == 42
assert Node(42).count() == 1
print("Node total/count: OK")

### Exercise 4.2 — Finish the TODOs

Extend your `Node` class (copy it down and add methods):

- `min_val()` / `max_val()` — smallest / largest value in the subtree.
- `find(x)` — `True` if `x` appears anywhere in the subtree.
- `max_depth()` — length of the longest root-to-leaf path (a lone node has depth 1).
  Careful: the naive `max(self.left.max_depth(), self.right.max_depth()) + 1`
  crashes on a leaf — handle `None` children.
- `apply(func)` — replace **every** node's `val` with `func(val)`, in place.

**Write your solution in `exercises/ex_4_2_node.py`**

In [ ]:
class Node:
    pass  # replace this (copy from above, then add the new methods)


tree = Node(10,
            Node(4, Node(1), Node(11)),
            Node(5, Node(15), Node(16)))
assert tree.min_val() == 1
assert tree.max_val() == 16
assert tree.find(11) == True
assert tree.find(99) == False
assert tree.max_depth() == 3
assert Node(7).max_depth() == 1

tree.apply(lambda v: v * 2)
assert tree.total() == 124
assert tree.min_val() == 2
print("Node methods: OK")

---
## Part 5 — A Decision Tree Is an Object

You get a job offer. Do you take it? Your (very rational) rules:

1. If the salary score is below 1 → **No**.
2. Otherwise, if the learning opportunity score is below 1 → **No**.
3. Otherwise, if the distance score is below 1 → **Yes**; too far → **No**.

That is a tree of decisions — and you can build it from objects.

### Exercise 5.1 — `DecisionNode`, `Yes`, `No`

- `DecisionNode(criteria, boundary, left, right)` stores all four.
  Its method `answer(circumstance)` looks up `circumstance[self.criteria]`
  (circumstance is a dict) and delegates to one of its children. How should
  the node decide which child to ask? What role does the `boundary` play?
- `Yes(DecisionNode)` and `No(DecisionNode)` are leaf classes: their `answer`
  ignores the circumstance and returns `True` / `False`. Give them their own
  no-argument `__init__`.

No `if`-chains about salaries anywhere — the *structure* of the objects encodes
the rules.

**Write your solution in `exercises/ex_5_1_decisionnode.py`**

In [ ]:
class DecisionNode:
    pass  # replace this

class Yes(DecisionNode):
    pass  # replace this

class No(DecisionNode):
    pass  # replace this


# the offer tree: salary -> learningOpp -> dist
dist = DecisionNode("dist", 1, Yes(), No())            # close enough -> Yes
learning = DecisionNode("learningOpp", 1, No(), dist)  # no learning -> No
offer_tree = DecisionNode("salary", 1, No(), learning) # low salary -> No

assert offer_tree.answer({"salary": 2,   "learningOpp": 3, "dist": 0.5}) == True
assert offer_tree.answer({"salary": 0.5, "learningOpp": 3, "dist": 0.5}) == False
assert offer_tree.answer({"salary": 2,   "learningOpp": 0, "dist": 0.5}) == False
assert offer_tree.answer({"salary": 2,   "learningOpp": 3, "dist": 5})   == False
print("DecisionNode: OK")

### This is a real decision tree

You built its structure by hand. The *Decision Trees* chapter later in this book
answers the natural next question: given a table of past decisions, how can `fit`
discover the right criteria and boundaries **automatically**? Same object, learned
instead of hand-crafted.

---
## Part 6 — Saving Objects to Files

Your program built a useful tree — then it exits, and the tree is gone. A model
you cannot save is a model you must retrain. Let's serialize it.

JSON can store dicts, lists, numbers, strings and booleans — but not your objects.
So the plan is: **object → dict → JSON file**, and back.

### Exercise 6.1 — `to_dict` and `from_dict`

- `to_dict(node)` — a `Yes` leaf becomes `{"leaf": True}`, a `No` leaf becomes
  `{"leaf": False}`, and an inner node becomes
  `{"criteria": ..., "boundary": ..., "left": <dict>, "right": <dict>}`
  (children converted recursively). **Hint:** `isinstance(node, Yes)` tells leaves apart.
- `from_dict(d)` — the exact inverse, rebuilding real objects.

The test saves the offer tree to a real file, loads it back, and checks that the
reloaded tree gives the same answers.

**Write your solution in `exercises/ex_6_1_to_dict.py`**

In [ ]:
import json

def to_dict(node):
    pass  # replace this

def from_dict(d):
    pass  # replace this


d = to_dict(offer_tree)
assert d["criteria"] == "salary"
assert d["boundary"] == 1
assert d["left"] == {"leaf": False}

with open("offer_tree.json", "w") as f:
    json.dump(d, f)
with open("offer_tree.json") as f:
    loaded = from_dict(json.load(f))

for case in [{"salary": 2,   "learningOpp": 3, "dist": 0.5},
             {"salary": 0.5, "learningOpp": 3, "dist": 0.5},
             {"salary": 2,   "learningOpp": 0, "dist": 0.5},
             {"salary": 2,   "learningOpp": 3, "dist": 5}]:
    assert loaded.answer(case) == offer_tree.answer(case)
print("save/load: OK")

### Aside — `pickle`

Python's `pickle` module can serialize almost any object in one line
(`pickle.dump(offer_tree, f)`), no `to_dict` needed. So why bother? A pickle file
is Python-only, unreadable by humans, and unsafe to load from untrusted sources.
Your JSON file is none of those things. Real ML tooling makes the same trade-off:
scikit-learn models ship as pickles (via `joblib`), while model *configs* travel
as JSON or YAML.

---
## Bonus Challenge — `FitPlane`

`TwoPointLine` fits a line through 2 points. In 3D, **three** points determine a
plane `z = a*x + b*y + d` — and finding `a`, `b`, `d` means solving three linear
equations:

```
x1*a + y1*b + 1*d = z1
x2*a + y2*b + 1*d = z2
x3*a + y3*b + 1*d = z3
```

You already own a tool for that: paste your `solve_equations` (and its helpers)
from the *Loops and Arrays* chapter into the cell below, then write `FitPlane`:

- `fit(points)` — `points` is a list of three `(x, y, z)` tuples. Build the three
  equations `[x, y, 1, z]` and solve for `(a, b, d)`.
- `predict(points)` — a list of `(x, y)` tuples in, a list of `a*x + b*y + d` out.

In [ ]:
# Paste your solve_equations (+ helpers) from the Loops and Arrays chapter here,
# then write FitPlane.

class FitPlane:
    pass  # replace this


plane = FitPlane()
plane.fit([(1, 1, 6), (2, 1, 8), (1, 2, 9)])   # these lie on z = 2x + 3y + 1
assert plane.predict([(2, 2)]) == [11.0]
assert plane.predict([(0, 0), (1, 1)]) == [1.0, 6.0]
print("FitPlane: OK")

---
## Summary

| Part | You built | The idea |
|------|-----------|----------|
| 1 | `Customer` | `__init__`, `self`, methods — data + behavior in one place |
| 2 | `TwoPointLine` | **the `fit`/`predict` model API**, input validation |
| 3 | `Animal` → `Dog`, `Cat` | inheritance |
| 4 | `Node.total/count/min_val/max_val/find/max_depth/apply` | recursion on object trees |
| 5 | `DecisionNode`, `Yes`, `No` | a decision tree as objects; leaves via inheritance |
| 6 | `to_dict` / `from_dict` | serialization — models that survive the program |
| Bonus | `FitPlane` | fit = solving equations (three points → a plane) |

From here on, everything is objects: the *Ancient Secrets of Prediction* chapter
finds best-fit models when no exact fit exists, *Gradient Descent* makes `fit`
iterative, and *Decision Trees* learns Part 5's tree from data.